# ***** WORKFLOW POUR IMPLEMENTER RAG SUR UN LLM ****

# 1 - Extraction (PDF --> Markdowns) via Docling

In [1]:
# # Le RAG (Retrieval Augmented Recognition est utilisé dans 90% des cas pour "fine-tuner" un modèle LLM classique (Claude, OpenAI etc)). 
# # Le modèle va "se nourrir" des données des pdf (factures, devis etc) pour devenir plus entraîné sur des données propres à l'entreprise

# ### Quel est le workflow pour implémenter un RAG sur un LLM ?

# # 1 - Extraction (PDF --> Texte): on utilisera l'outil Docling (IBM, open source ) qui est parfait pour les documents qui comportent des données tabulaires (devis, factures) ce qui est le cas pour Paysagio.

# #    ------ >      Docling transforme les PDF en markdowns propres, pour l'IA c'est banger     <---------

from docling.document_converter import DocumentConverter

def convertir_pdf_pro(chemin_pdf):
    # 1. Initialiser le convertisseur
    converter = DocumentConverter()
    
    # 2. Lancer la conversion (Docling analyse tout : mise en page, tableaux...)
    resultat = converter.convert(chemin_pdf)
    
    # 3. Exporter le résultat en Markdown (le format préféré des IA)
    texte_markdown = resultat.document.export_to_markdown()
    
    return texte_markdown

# Teste-le sur un devis complexe !
mon_devis= convertir_pdf_pro("FACTURE 1100481.pdf")
print(mon_devis)









/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 1553.18it/s]


## BICY-STORE  - BESANCON

14 RUE LUC BRETON

## 25000 BESANCON

TEL :

03.81.85.93.52

E-MAIL :

besancon@bicy-store.com

REG :

81394662100063 / APE 4511Z

TVA :

FR93 813 946 621

SARL AU CAPITAL DE 2 000

<!-- image -->

N  CLIENT 1100380

OPERATEUR XAVIER K.

DATE 14/03/2026

PAGE 1/1

1100481 FACTURE

REFERENCE

QUANTITE

P.U. TTC

REM. %

MONTANT TTC

TVA

P.U. HT

DESIGNATION

CONDITION ACCORD E PAIEMENT COMPTANT

HEURE 11:09

CH ANCE 14/03/2026

## RESERVATION N 1300048 DU 12/03/2026

BOBIKE GO MAXI CARRIER

*2

1,00

74,92

0,00

100,00 89,90

20,00

DUMA 4.1 UNIV SIZE M DEEP BLUE IPO 730

32758

1,00

1 749,17

2 099,00 20,00

2 099,00

N  de sØrie : O2000970

ModŁle : DUMA 4.1 UNIV SIZE M DEEP BLUE

IPO 730

NumØro de moteur : M60325270977

Couleur : BLEU

N  de clef : 1274

Etat : Neuf Moteur : ANANDA 70NM KilomØtrage : 0 Taille de cadre : M/L AnnØe modŁle : 2026

Marque : O2FEEL

Batterie : A2599D0006621

Puissance : 250W

CatØgorie : VTC

Marquage Bicycod : BC3A3CJE8C

M

# 2 - Chunking avec LangChain

In [2]:
# # 2 - Chunking : On crée une fonction python qui va découper les phrases en bloc de mots 

from langchain_text_splitters import MarkdownHeaderTextSplitter

# 1. On définit les niveaux de titres qui servent de "coupe"
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]




In [3]:
# 2. Ton texte récupéré via Docling
# (Imagine que resultat_markdown est le string que Docling t'a donné)

result_markdown = mon_devis

In [4]:
# 3. On initialise le splitter
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# 4. On découpe !
md_header_splits = markdown_splitter.split_text(result_markdown)

In [5]:
# On boucle dessus pour affichage du résultat

for i, chunk in enumerate(md_header_splits):
    print(f"--- CHUNK {i} ---")
    print(f"Contenu : {chunk.page_content[:100]}...")
    print(f"Métadonnées : {chunk.metadata}") # Ici, l'IA sait de quel titre vient le texte !


--- CHUNK 0 ---
Contenu : 14 RUE LUC BRETON...
Métadonnées : {'Header 2': 'BICY-STORE  - BESANCON'}
--- CHUNK 1 ---
Contenu : TEL :  
03.81.85.93.52  
E-MAIL :  
besancon@bicy-store.com  
REG :  
81394662100063 / APE 4511Z  
T...
Métadonnées : {'Header 2': '25000 BESANCON'}
--- CHUNK 2 ---
Contenu : BOBIKE GO MAXI CARRIER  
*2  
1,00  
74,92  
0,00  
100,00 89,90  
20,00  
DUMA 4.1 UNIV SIZE M DEEP...
Métadonnées : {'Header 2': 'RESERVATION N 1300048 DU 12/03/2026'}
--- CHUNK 3 ---
Contenu : - FRANCE...
Métadonnées : {'Header 2': 'MONSIEUR FANDINO LUCA'}


# 3 - Embedding (via modèle gratuit HuggingFace) et stockage ChromaDB 

In [6]:
%pip install langchain-huggingface
%pip install sentence-transformers
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# --> on transfomrme les chunks en vecteurs puis on les stocke dans une BDD vectorielle locale, ChromaDB)
try:
    import langchain_huggingface
    print("🚀 VICTOIRE ! Le module est reconnu.")
except Exception as e:
    print(f"💀 Toujours pas... Erreur : {e}")

# 1. On initialise le modèle HuggingFace
# La première fois, il va télécharger le modèle sur ton PC (environ 400Mo)
# Les fois suivantes, il le chargera instantanément depuis ton disque.
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model_kwargs = {'device': 'cpu'} # Utilise 'cuda' si tu as une carte graphique Nvidia
encode_kwargs = {'normalize_embeddings': False}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
🚀 VICTOIRE ! Le module est enfin reconnu.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8612.09it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# 2. On crée la base Chroma avec tes chunks Docling
# (md_header_splits est la liste de chunks que tu as déjà générée)
vector_db = Chroma.from_documents(
    documents=md_header_splits, 
    embedding=embeddings,
    persist_directory="./db_paysagio_local"
)

print("✅ Base locale ChromaDB créée avec succès via HuggingFace !")

✅ Base locale créée avec succès via HuggingFace !


# On met en place la QA_CHAIN final avec le LLM (Ollama)

In [9]:
# # 4 - Retrieval & LLM (La Question)

from langchain_community.chat_models import ChatOllama                                                                                                                                                            
from langchain_core.prompts import ChatPromptTemplate                                                                                                                                                             
from langchain_core.output_parsers import StrOutputParser                                                                                                                                                         
from langchain_core.runnables import RunnablePassthrough                                                                                                                                                          
                                                                                                                                                                                                                    
# 1. LLM local                                                                                                                                                                                                    
llm = ChatOllama(model="llama3", temperature=0)                                                                                                                                                                   
                                                                                                                                                                                                                
# 2. Retriever depuis votre base ChromaDB                                                                                                                                                                         
retriever = vector_db.as_retriever(search_kwargs={
        "k": 3,
        #"filter": {"societe_id": user_societe_id} # L'isolation se joue ici !
    })                                                                                                                                                        
                                                                                                                                                                                                                
# 3. Prompt                                                                                                                                                                                                       
prompt = ChatPromptTemplate.from_template("""                                                                                                                                                                     
Réponds à la question en te basant uniquement sur le contexte suivant :                                                                                                                                           
                                                                                                                                                                                                                
{context}                                                                                                                                                                                                         
                                                                                                                                                                                                                
Question : {question}                                                                                                                                                                                             
""")                                                                                                                                                                                                            
                                                                                                                                                                                                                
# 4. Chaîne RAG                                                                                                                                                                                                   
def format_docs(docs):                                                                                                                                                                                            
    return "\n\n".join(doc.page_content for doc in docs)                                                                                                                                                          
                                                                                                                                                                                                                
rag_chain = (                                                                                                                                                                                                     
    {"context": retriever | format_docs, "question": RunnablePassthrough()}                                                                                                                                       
    | prompt                                                                                                                                                                                                      
    | llm                                                                                                                                                                                                         
    | StrOutputParser()                                                                                                                                                                                           
)                                                                                                                                                                                                                 
                                                                                                                                                                                                                
# 5. Test                                                                                                                                                                                                         
reponse = rag_chain.invoke("Quels sont les produits vendus ici?")                                                                                                                                                  
print(reponse)   

Selon le contexte, les produits vendus sont :

* BOBIKE GO MAXI CARRIER (2 pièces)
* DUMA 4.1 UNIV SIZE M DEEP BLUE IPO 730 (un seul produit)

Ces deux produits font partie d'une facture émise par la société "BESANCON@BICY-STORE.COM" le 14 mars 2026.


In [9]:
################# FONCTIONNEMENT DE LA RAG_CHAIN #################

# 1 . {"context": retriever | format_docs, ...} :

# On prépare les ingrédients. On envoie la question dans le retriever.

# Le résultat (des objets "Document") passe dans format_docs pour devenir une simple chaîne de texte propre.



# 2 . "question": RunnablePassthrough() :

# Cela veut dire : "Laisse passer la question originale telle quelle, sans la modifier", pour qu'elle arrive aussi au prompt.



# 3 . | prompt :

# On injecte le contexte et la question dans ton moule de texte.



# 4 . | llm :

# Le texte complet (Instructions + Contexte + Question) est envoyé à Llama 3.



# 5. | StrOutputParser() :

# Par défaut, l'IA renvoie un objet complexe avec des métadonnées. Ce parser nettoie tout pour ne te donner que le texte brut de la réponse.